In [26]:
import json
import pandas as pd

In [27]:
DATA_PATH = "data/ori_pqal.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

type(data)

dict

In [28]:
len(data)

1000

In [29]:
list(data.keys())[:5]

['21645374', '16418930', '9488747', '17208539', '10808977']

In [30]:
first_key = list(data.keys())[0]

first_key

'21645374'

In [31]:
data[first_key]

{'QUESTION': 'Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?',
 'CONTEXTS': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.',
  'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD)

In [32]:
rows = []

for question_id, item in data.items():
    row = {
        "question_id": question_id,
        "question": item.get("QUESTION", ""),
        "contexts": item.get("CONTEXTS", []),
        "labels": item.get("LABELS", []),
        "meshes": item.get("MESHES", []),
        "year": item.get("YEAR", ""),
        "reasoning_required": item.get("reasoning_required_pred", ""),
        "final_decision": item.get("final_decision", ""),
        "long_answer": item.get("LONG_ANSWER", "")
    }
    
    rows.append(row)

df = pd.DataFrame(rows)

df.head()

,question_id,question,contexts,labels,meshes,year,reasoning_required,final_decision,long_answer
0,21645374,Do mitochondria play a role in remodelling lac...,[Programmed cell death (PCD) is the regulated ...,"[BACKGROUND, RESULTS]","[Alismataceae, Apoptosis, Cell Differentiation...",2011,yes,yes,Results depicted mitochondrial dynamics in viv...
1,16418930,Landolt C and snellen e acuity: differences in...,[Assessment of visual acuity depends on the op...,"[BACKGROUND, PATIENTS AND METHODS, RESULTS]","[Adolescent, Adult, Aged, Aged, 80 and over, A...",2006,no,no,"Using the charts described, there was only a s..."
2,9488747,"Syncope during bathing in infants, a pediatric...",[Apparent life-threatening events in infants a...,"[BACKGROUND, CASE REPORTS]","[Baths, Histamine, Humans, Infant, Syncope, Ur...",1997,yes,yes,"""Aquagenic maladies"" could be a pediatric form..."
3,17208539,Are the long-term results of the transanal pul...,[The transanal endorectal pull-through (TERPT)...,"[PURPOSE, METHODS, RESULTS]","[Child, Child, Preschool, Colectomy, Female, H...",2007,yes,no,Our long-term study showed significantly bette...
4,10808977,Can tailored interventions increase mammograph...,[Telephone counseling and tailored print commu...,"[BACKGROUND, DESIGN, PARTICIPANTS, INTERVENTIO...","[Cost-Benefit Analysis, Female, Health Mainten...",2000,yes,yes,The effects of the intervention were most pron...


In [33]:
df.shape

(1000, 9)

In [34]:
df.columns

Index(['question_id', 'question', 'contexts', 'labels', 'meshes', 'year',
       'reasoning_required', 'final_decision', 'long_answer'],
      dtype='object')

In [35]:
df.head(3)

,question_id,question,contexts,labels,meshes,year,reasoning_required,final_decision,long_answer
0,21645374,Do mitochondria play a role in remodelling lac...,[Programmed cell death (PCD) is the regulated ...,"[BACKGROUND, RESULTS]","[Alismataceae, Apoptosis, Cell Differentiation...",2011,yes,yes,Results depicted mitochondrial dynamics in viv...
1,16418930,Landolt C and snellen e acuity: differences in...,[Assessment of visual acuity depends on the op...,"[BACKGROUND, PATIENTS AND METHODS, RESULTS]","[Adolescent, Adult, Aged, Aged, 80 and over, A...",2006,no,no,"Using the charts described, there was only a s..."
2,9488747,"Syncope during bathing in infants, a pediatric...",[Apparent life-threatening events in infants a...,"[BACKGROUND, CASE REPORTS]","[Baths, Histamine, Humans, Infant, Syncope, Ur...",1997,yes,yes,"""Aquagenic maladies"" could be a pediatric form..."


In [36]:
def join_contexts(contexts):
    if isinstance(contexts, list):
        return " ".join(contexts)
    return str(contexts)

df["context_text"] = df["contexts"].apply(join_contexts)

In [37]:
df[["question", "context_text", "final_decision", "long_answer"]].head()

,question,context_text,final_decision,long_answer
0,Do mitochondria play a role in remodelling lac...,Programmed cell death (PCD) is the regulated d...,yes,Results depicted mitochondrial dynamics in viv...
1,Landolt C and snellen e acuity: differences in...,Assessment of visual acuity depends on the opt...,no,"Using the charts described, there was only a s..."
2,"Syncope during bathing in infants, a pediatric...",Apparent life-threatening events in infants ar...,yes,"""Aquagenic maladies"" could be a pediatric form..."
3,Are the long-term results of the transanal pul...,The transanal endorectal pull-through (TERPT) ...,no,Our long-term study showed significantly bette...
4,Can tailored interventions increase mammograph...,Telephone counseling and tailored print commun...,yes,The effects of the intervention were most pron...


In [38]:
df["medical_text"] = (
    "Question: " + df["question"].astype(str) + "\n\n" +
    "Context: " + df["context_text"].astype(str) + "\n\n" +
    "Long answer: " + df["long_answer"].astype(str) + "\n\n" +
    "Final decision: " + df["final_decision"].astype(str)
)

In [39]:
df = df.dropna(subset=["question", "context_text"])

df = df[df["question"].str.len() > 5]
df = df[df["context_text"].str.len() > 50]

df.shape

(1000, 11)

In [41]:
df.to_csv("data/pubmedqa_clean.csv", index=False, encoding="utf-8")

print("Saved clean dataset to ../data/pubmedqa_clean.csv")

Saved clean dataset to ../data/pubmedqa_clean.csv
